# Shor: compiled order finding for N=15

PHYS 3810, session 3: **20-minute core** and **10-minute optional expansion** after the HHL example. These are proposed timings, pending rehearsal and instructor feedback. Before class, run all cells. In class, restart and clear outputs, make each prediction, then execute the corresponding cells.

This is an explicitly **compiled full-domain N=15 example**, not a general factoring implementation. Four residue qubits encode every integer 0 through 15. Dense permutation matrices and exact small orders are disclosed conveniences. The existing QPE helper accepts only one/two target qubits; this lesson uses a **separate order-finding builder** and reuses canonical inverse QFT. No hardware, accounts, or credentials are needed.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "src" / "phys3810" / "shor.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Keep the extracted course folder together and start Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Core, minutes 0-5: predict the orbit and registers

For a base a coprime to 15, $M_a|x\rangle=|ax\bmod15\rangle$ for x<15, while 15 stays fixed. Both **0 and 15 are fixed**, with coefficient +1. Complementing all four bits for a=14 would incorrectly swap those states, even though it works on the orbit of 1.

Start all m+4 qubits at zero. Count uses qubits 0 through m−1; residue uses the next four. The source prepares residue **0001**, puts count into a uniform superposition, applies controlled multiplication constants $a^{2^j}\bmod15$, then inverse QFT with swaps. Statevector order is **residue tensor count**. Measure count[j] into phase[j]; decode `int(bits,2)/2**m`, without reversing strings.

Predict the orbit of 1 under multiplication by 2 modulo 15. After how many multiplications does it return to 1? Which powers need a nonidentity gate? Four default **counting** bits and four **residue** bits serve different purposes.

In [ ]:
from fractions import Fraction
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import ClassicalRegister, transpile
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from src.phys3810.shor import (
    classical_factor_15, factors_from_order_15, order_finding_15, recover_order_15,
)

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0], "| Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)
ATOL, SHOTS, SEED = 2e-11, 2048, 2026


def joint_reference(trial_base, width):
    """Classical validation only: modular powers and a negative Fourier sum."""
    size = 2**width
    reference = np.zeros((16, size), dtype=complex)
    for exponent in range(size):
        reference[pow(trial_base, exponent, 15)] += np.exp(
            -2j*np.pi*exponent*np.arange(size)/size) / size
    return reference


def sample_trial(trial_circuit, trial_base, width, probabilities):
    """One finite batch; None means an honest inconclusive or unsuccessful trial."""
    measured = trial_circuit.copy()
    phase = ClassicalRegister(width, "phase")
    measured.add_register(phase)
    measured.measure(range(width), phase)
    result = StatevectorSampler(seed=SEED).run([measured], shots=SHOTS).result()[0]
    counts = result.data.phase.get_counts()
    assert sum(counts.values()) == SHOTS
    support = {format(y, f"0{width}b") for y, p in enumerate(probabilities) if p > ATOL}
    assert not set(counts)-support, "Unexpected ideal phase bin."
    for bits in support:
        p = float(probabilities[int(bits, 2)])
        bound = 6*np.sqrt(p*max(0, 1-p)/SHOTS) + 1/SHOTS
        assert abs(counts.get(bits, 0)/SHOTS-p) <= bound
    rows = []
    order_shots = factor_shots = 0
    for bits, count in sorted(counts.items()):
        fraction = Fraction(int(bits, 2), 2**width)
        recovered = recover_order_15(trial_base, bits)
        factors = None if recovered is None else factors_from_order_15(trial_base, recovered)
        rows.append((bits, count, fraction, recovered, factors))
        order_shots += count if recovered is not None else 0
        factor_shots += count if factors is not None else 0
    assert trial_circuit.num_clbits == 0
    return {"counts": counts, "rows": rows, "order_shots": order_shots,
            "factor_shots": factor_shots, "shots": SHOTS}

In [ ]:
# Editable inputs: base in {1,2,4,7,8,11,13,14}, counting width 4..6.
base = 2
num_counting_qubits = 4
circuit = order_finding_15(base, num_counting_qubits=num_counting_qubits)
state = Statevector.from_instruction(circuit)
size = 2**num_counting_qubits
assert circuit.num_qubits == num_counting_qubits+4 and circuit.num_clbits == 0
print("Registers:", [(r.name, len(r)) for r in circuit.qregs])
print("Powered constants:", [pow(base, 2**j, 15) for j in range(num_counting_qubits)])
figure = circuit.draw("mpl", fold=35)
display(figure)
plt.close(figure)

## Core, minutes 5-12: predict the phase distribution

For the default base 2, use the orbit length you predicted. Which four-bit strings should occur, and with what probabilities? The initial residue state is a superposition of multiplier eigenstates, so several phase outcomes are expected.

The independent expected joint amplitude is
$$\psi[z,y]=\frac1Q\sum_{x=0}^{Q-1}e^{-2\pi ixy/Q}[z=a^x\bmod15],\qquad Q=2^m.$$
This is a classical test reference, not circuit construction. It checks complex amplitudes and register order; a visually plausible histogram alone is insufficient. The exact small orders divide four. This explains the dyadic phase peaks here, not a general precision guarantee for Shor.

In [ ]:
expected_joint = joint_reference(base, num_counting_qubits)
actual_joint = state.data.reshape(16, size)
np.testing.assert_allclose(actual_joint, expected_joint, rtol=0, atol=ATOL)
np.testing.assert_allclose(np.vdot(state.data, state.data), 1, rtol=0, atol=ATOL)
expected_probabilities = np.sum(abs(expected_joint)**2, axis=0)
actual_probabilities = np.sum(abs(actual_joint)**2, axis=0)
np.testing.assert_allclose(actual_probabilities, expected_probabilities, rtol=0, atol=ATOL)
print("Expected phase bins:", {format(y, f"0{num_counting_qubits}b"): round(float(p), 8)
                              for y, p in enumerate(expected_probabilities) if p > ATOL})
print("PASSED: complete complex state, norm, and counting marginal")

## Core, minutes 12-20: verify a denominator, then take GCDs

Before running, predict which outcomes contain enough information to recover the order. Does zero phase give an order? Could a reduced fraction hide part of it? Is a valid order always enough to obtain factors?

For this exact-grid model, reduce y/Q exactly. Reject zero; verify the denominator is a period and is minimal. **Do not try multiples, use an order table, or approximate an off-support fraction.** An insufficient denominator returns `None`. Given a verified even order r, compute $x=a^{r/2}\bmod15$, reject x=±1 modulo 15, and take gcd(x−1,15) and gcd(x+1,15). The two functions report order and factor success separately. The core uses one 2048-shot local V2 batch, seed 2026, with no retry loop.

In [ ]:
trial = sample_trial(circuit, base, num_counting_qubits, expected_probabilities)
print("bits | shots | exact phase | verified order | factors")
for bits, count, fraction, recovered, factors in trial["rows"]:
    print(bits, count, str(fraction), recovered, factors, sep=" | ")
print(f"Order recovery: {trial['order_shots']}/{SHOTS}; factors: {trial['factor_shots']}/{SHOTS}")
if trial["factor_shots"] == 0:
    print("This batch produced no factors. An unsuccessful result is retained; no automatic retry.")
labels = [format(y, f"0{num_counting_qubits}b") for y in range(size)]
frequencies = np.array([trial["counts"].get(bits, 0)/SHOTS for bits in labels])
figure, axes = plt.subplots(figsize=(9, 3.5))
axes.bar(range(size), frequencies, label="Local V2 samples", alpha=0.7)
axes.plot(range(size), expected_probabilities, "ko", markersize=3, label="Independent expectation")
ticks = np.arange(0, size, max(1, size//16))
axes.set_xticks(ticks, [labels[y] for y in ticks], rotation=45)
axes.set(xlabel="Counting bits (no reversal)", ylabel="Probability", ylim=(0, 1.05),
         title=f"N=15, base={base}, m={num_counting_qubits}")
axes.legend()
figure.tight_layout()
display(figure)
plt.close(figure)
print("PASSED: shot accounting, allowed bins, and six-standard-error bounds")

## Optional expansion, 10 minutes: separate three kinds of failure

Predict before executing:

- For base 2, does `1000` identify the true order? What happens to off-support `0010`, whose denominator is a nonminimal period?
- For base 11, the useful phase is 1/2. Compute the two GCDs explicitly; returning x−1 and x+1 directly would be wrong.
- For base 14, the useful phase is also 1/2. Can order recovery succeed while factor extraction fails?
- For noncoprime base 6, what does the classical GCD precheck return? That is a classical shortcut, not a quantum result.

The next cell uses exactly one batch each for bases 11 and 14, at the selected width. Together with the core this is **three batches, 6144 shots**. Change the core base/width only within the stated contract, then rerun its cell and subsequent cells. Base 1 deliberately gives only zero phase and no recovered order. All expansion code executes during validation; the time allocation is optional, not execution.

In [ ]:
expansion_results = {}
for trial_base in (11, 14):
    trial_circuit = order_finding_15(trial_base, num_counting_qubits=num_counting_qubits)
    reference = joint_reference(trial_base, num_counting_qubits)
    np.testing.assert_allclose(Statevector.from_instruction(trial_circuit).data,
                               reference.ravel(), rtol=0, atol=ATOL)
    probabilities = np.sum(abs(reference)**2, axis=0)
    summary = sample_trial(trial_circuit, trial_base, num_counting_qubits, probabilities)
    expansion_results[trial_base] = summary
    print(f"base={trial_base}: order recovery {summary['order_shots']}/{SHOTS}, factors {summary['factor_shots']}/{SHOTS}")
    for row in summary["rows"]:
        print(row)
assert trial["shots"] + sum(s["shots"] for s in expansion_results.values()) == 6144
figure, axes = plt.subplots(figsize=(6, 3.5))
positions = np.arange(2)
axes.bar(positions-0.18, [expansion_results[b]["order_shots"]/SHOTS for b in (11, 14)],
         width=0.36, label="Order recovered")
axes.bar(positions+0.18, [expansion_results[b]["factor_shots"]/SHOTS for b in (11, 14)],
         width=0.36, label="Factors obtained")
axes.set_xticks(positions, ["base 11", "base 14"])
axes.set(ylabel="Fraction of all shots", ylim=(0, 1.05), title="An order need not yield factors")
axes.legend()
figure.tight_layout()
display(figure)
plt.close(figure)

In [ ]:
assert recover_order_15(2, "0000") is None
assert recover_order_15(2, "1000") is None  # Reduced denominator 2 hides the true order.
assert recover_order_15(2, "0010") is None  # Period 8 is not minimal.
assert recover_order_15(11, "1000") == 2
assert factors_from_order_15(11, 2) == (3, 5)
assert recover_order_15(14, "1000") == 2
assert factors_from_order_15(14, 2) is None
assert factors_from_order_15(1, 1) is None
assert classical_factor_15(6) == (3, 5)
try:
    order_finding_15(6)
except ValueError as error:
    print("PASSED: noncoprime quantum base rejected:", error)
else:
    raise AssertionError("A nonbijective modular multiplier was accepted.")
print("PASSED: zero/inconclusive/nonminimal phases, trivial roots, and separate classical GCD")

## Resources and limitations

This demonstration uses 8-10 logical qubits, dense 16x16 multiplier matrices (32x32 with control), and at most two nonidentity controlled multipliers. Constants equal to 1 are omitted by their value, not by a known-order table. The count $2^m-1$ describes naive repeated access to one multiplier; it is not the number of compiled blocks here. Dense storage/synthesis scales exponentially with residue width.

The diagnostic below transpiles the selected circuit into `u`/`cx`, optimization level 1, seed 2026, with **no hardware target or connectivity restriction**. It records actual local synthesis counts and checks the prepared state again. These counts are not hardware performance, and the fixed modulus/exact dyadic orders are not a general Shor speedup or a practical cryptographic attack. General arithmetic networks and approximate continued-fraction recovery are outside this lesson.

In [ ]:
compiled = transpile(circuit, basis_gates=["u", "cx"], optimization_level=1, seed_transpiler=SEED)
np.testing.assert_allclose(Statevector.from_instruction(compiled).data,
                           expected_joint.ravel(), rtol=0, atol=ATOL)
resource_counts = {"qubits": compiled.num_qubits, "depth": compiled.depth(),
                   "operations": dict(compiled.count_ops())}
print("Local u/cx synthesis, optimization_level=1, seed_transpiler=2026:", resource_counts)
print("PASSED: synthesized circuit preserves the complete prepared state")

Read the [bounded design contract](../SHOR_PLAN.md) and reusable source under `src/phys3810/shor.py`. Mathematical references: [Shor's original paper](https://arxiv.org/abs/quant-ph/9508027), [IBM's modular-permutation explanation](https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm), and [general order recovery](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/shor-algorithm). The tutorial's hardware/deprecated-wrapper examples are not used here. Our conservative exact-rational recovery deliberately reports inconclusive samples instead of adding a general recovery algorithm.